## Example 1: Dispatch a JSON-RPC 2.0 call to a Python function

In [ ]:
import importlib.metadata
import json

from jsonrpc import Dispatcher, JSONRPCResponseManager

# Example 1: register methods and answer a request string
assert importlib.metadata.version("json-rpc") == "1.15.0"
dispatcher = Dispatcher()
dispatcher.add_method(lambda a, b: a + b, name="add")
request = json.dumps({"jsonrpc": "2.0", "method": "add", "params": [2, 3], "id": 1})
response = JSONRPCResponseManager.handle(request, dispatcher)
assert response.data == {"jsonrpc": "2.0", "result": 5, "id": 1}, response.data
print("Example 1 OK:", response.json)

## Example 2: Errors come back as JSON-RPC error objects

In [ ]:
import json

from jsonrpc import Dispatcher, JSONRPCResponseManager

# Example 2: unknown method, bad params and invalid JSON each map to their standard error codes
dispatcher = Dispatcher({"echo": lambda text: text})
def call(raw):
    return JSONRPCResponseManager.handle(raw, dispatcher).data["error"]["code"]
assert call(json.dumps({"jsonrpc": "2.0", "method": "nope", "id": 1})) == -32601
assert call(json.dumps({"jsonrpc": "2.0", "method": "echo", "params": [1, 2], "id": 2})) == -32602
assert call("{not json") == -32700
print("Example 2 OK: -32601, -32602, -32700")

## Example 3: Batch requests and notifications

In [ ]:
import json

from jsonrpc import Dispatcher, JSONRPCResponseManager

# Example 3: a batch answers each call in order; a notification (no id) gets no answer
seen = []
dispatcher = Dispatcher({"square": lambda x: x * x, "log": lambda msg: seen.append(msg)})
batch = [
    {"jsonrpc": "2.0", "method": "square", "params": [3], "id": 1},
    {"jsonrpc": "2.0", "method": "log", "params": ["hello"]},
    {"jsonrpc": "2.0", "method": "square", "params": {"x": 4}, "id": 2},
]
response = JSONRPCResponseManager.handle(json.dumps(batch), dispatcher)
assert [r["result"] for r in response.data] == [9, 16], response.data
assert seen == ["hello"], seen
print("Example 3 OK:", response.data)

## Example 4: Build requests and responses as objects

In [ ]:
from jsonrpc.jsonrpc2 import JSONRPC20Request, JSONRPC20Response

# Example 4: the request and response classes validate and serialize the 2.0 envelope
request = JSONRPC20Request(method="sum", params=[1, 2, 3], _id=7)
assert request.data == {"jsonrpc": "2.0", "method": "sum", "params": [1, 2, 3], "id": 7}, request.data
parsed = JSONRPC20Request.from_json(request.json)
assert parsed.method == "sum" and parsed.params == [1, 2, 3] and parsed._id == 7
response = JSONRPC20Response(result=6, _id=parsed._id)
assert response.data == {"jsonrpc": "2.0", "result": 6, "id": 7}, response.data
print("Example 4 OK:", request.json, "->", response.json)

## Example 5: Version 1.0 requests are understood too

In [ ]:
import json

from jsonrpc import Dispatcher, JSONRPCResponseManager

# Example 5: a JSON-RPC 1.0 call (no "jsonrpc" member) gets a 1.0-shaped answer (no "jsonrpc" member either)
dispatcher = Dispatcher({"upper": lambda s: s.upper()})
response = JSONRPCResponseManager.handle(json.dumps({"method": "upper", "params": ["abc"], "id": 5}), dispatcher)
assert response.data == {"result": "ABC", "id": 5}, response.data
print("Example 5 OK:", response.json)